In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

tqdm.pandas()

def random_date(start_date, end_date):
    """Generate a random date between start_date and end_date."""
    delta = end_date - start_date
    random_days = np.random.randint(0, delta.days + 1)
    return start_date + timedelta(days=random_days)

np.random.seed(42)  
num_samples = 8000
lats = np.random.uniform(-90, 90, num_samples) 
lons = np.random.uniform(-180, 180, num_samples)  
start_date = datetime(2000, 1, 1)
end_date = datetime(2006, 12, 31)
dates = [random_date(start_date, end_date) for _ in range(num_samples)]

df = pd.DataFrame({
    'Latitude': lats,
    'Longitude': lons,
    'Sample Date': dates
})

def compute_Landsat_values(row):
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'])

    bbox_size = 0.00089831
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]
    try:
        catalog = pystac_client.Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=pc.sign_inplace,
        )

        search = catalog.search(
            collections=["landsat-c2-l2"],
            bbox=bbox,
            datetime="2000-01-01/2006-12-31",
            query={"eo:cloud_cover": {"lt": 10}},
        )
        
        time.sleep(0.5) 
        
        items = search.item_collection()
        if not items:
            return pd.Series({
                "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
            })
        
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])
        
        time.sleep(0.5)  
        
        bands_of_interest = ["green", "nir08", "swir16", "swir22"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)
        green = data["green"].astype("float")
        nir = data["nir08"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")
       
        median_green = float(green.median(skipna=True).values)
        median_nir = float(nir.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)
        median_green = median_green if median_green != 0 else np.nan
        median_nir = median_nir if median_nir != 0 else np.nan
        median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
        median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
       
        return pd.Series({
            "nir": median_nir,
            "green": median_green,
            "swir16": median_swir16,
            "swir22": median_swir22,
        })
   
    except Exception as e:
        print(f"Error for location {lat}, {lon}, date {date}: {str(e)}")
        return pd.Series({
            "nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan
        })


max_workers = 10 
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures_to_index = {executor.submit(compute_Landsat_values, row): index for index, row in df.iterrows()}
    results = [None] * len(df)
    for future in tqdm(as_completed(futures_to_index), total=len(df), desc="Processing samples"):
        index = futures_to_index[future]
        results[index] = future.result()

landsat_features = pd.concat(results, axis=1).T
df_with_features = pd.concat([df[['Latitude', 'Longitude', 'Sample Date']], landsat_features], axis=1)

eps = 1e-10
df_with_features['NDMI'] = (df_with_features['nir'] - df_with_features['swir16']) / (df_with_features['nir'] + df_with_features['swir16'] + eps)
df_with_features['MNDWI'] = (df_with_features['green'] - df_with_features['swir16']) / (df_with_features['green'] + df_with_features['swir16'] + eps)
df_with_features.to_csv('landsat_features_2000_2006_8000_samples.csv', index=False)

Processing samples:   2%|▏         | 140/8000 [01:45<29:47,  4.40it/s]  

Error for location -38.72871101205583, 162.33888667170035, date 2002-08-14 00:00:00: <!DOCTYPE html PUBLIC '-//W3C//DTD XHTML 1.0 Transitional//EN' 'http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd'>
<html xmlns='http://www.w3.org/1999/xhtml'>

<head>
    <meta content='text/html; charset=utf-8' http-equiv='content-type' />
    <style type='text/css'>
        body {
            font-family: Arial;
            margin-left: 40px;
        }

        img {
            border: 0 none;
        }

        #content {
            margin-left: auto;
            margin-right: auto
        }

        #message h1 {
            font-size: 24px;
            font-weight: normal;
            color: #000000;
            margin: 34px 0px 0px 0px
        }

        #message h2 {
            font-size: 20px;
            font-weight: normal;
            color: #000000;
            margin: 34px 0px 0px 0px
        }

        #message p {
            font-size: 16px;
            color: #000000;
       

Processing samples:  93%|█████████▎| 7412/8000 [1:50:43<10:23,  1.06s/it]  

Error for location 85.6567553010366, -89.37199285828906, date 2003-05-19 00:00:00: <!DOCTYPE html PUBLIC '-//W3C//DTD XHTML 1.0 Transitional//EN' 'http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd'>
<html xmlns='http://www.w3.org/1999/xhtml'>

<head>
    <meta content='text/html; charset=utf-8' http-equiv='content-type' />
    <style type='text/css'>
        body {
            font-family: Arial;
            margin-left: 40px;
        }

        img {
            border: 0 none;
        }

        #content {
            margin-left: auto;
            margin-right: auto
        }

        #message h1 {
            font-size: 24px;
            font-weight: normal;
            color: #000000;
            margin: 34px 0px 0px 0px
        }

        #message h2 {
            font-size: 20px;
            font-weight: normal;
            color: #000000;
            margin: 34px 0px 0px 0px
        }

        #message p {
            font-size: 16px;
            color: #000000;
         

Processing samples: 100%|██████████| 8000/8000 [1:59:11<00:00,  1.12it/s]


Error for location 4.915961215630795, 123.13396209798763, date 2006-07-26 00:00:00: Read failed. See previous exception for details.


In [2]:
import pandas as pd

df = pd.read_csv('landsat_features_2000_2006_8000_samples.csv')
pd.set_option('display.max_rows', None)
print(df.head())

    Latitude   Longitude Sample Date      nir    green   swir16   swir22  \
0 -22.582779   79.296453  2004-09-08      NaN      NaN      NaN      NaN   
1  81.128575   67.421882  2000-11-18  30598.0  35003.0  16916.5  16436.0   
2  41.758910 -145.528488  2002-04-02      NaN      NaN      NaN      NaN   
3  17.758527  152.126066  2001-06-26      NaN      NaN      NaN      NaN   
4 -61.916645   24.649993  2005-10-14      NaN      NaN      NaN      NaN   

       NDMI     MNDWI  
0       NaN       NaN  
1  0.287944  0.348357  
2       NaN       NaN  
3       NaN       NaN  
4       NaN       NaN  
